# Notebook 4 — Multi-pH Experiments: 1, 2, and 3 pH Conditions

HDX-MS experiments can be performed at one, two, or three pH conditions.
Adding more pH conditions extends the effective observation window:

| Condition | Exchange speed relative to pH 6 | Benefit |
|-----------|----------------------------------|---------|
| pH 5.0    | ~10× slower (1 log-unit below)   | Constrains **fast** rates that are already done at pH 6 |
| pH 6.0    | Reference                        | — |
| pH 9.0    | ~1000× faster (3 log-units above) | Constrains **slow** rates that never exchange at pH 6 |

The pipeline relates pH conditions via **merge factors** — log₁₀ scale parameters
inferred jointly with the exchange rates:

$$t^\mathrm{eff}_i = t^\mathrm{actual}_i \times 10^{\,\mathrm{sign}_i \times \mathrm{mf}_i}$$

where sign = +1 for pH above reference (faster exchange, longer effective exposure)
and −1 for pH below (slower exchange, shorter effective exposure).

**Goals:**
1. Run a single-pH experiment (pH 6.0) — the baseline
2. Add pH 9.0 — expands coverage towards **slow** rates
3. Add pH 5.0 as well — expands coverage towards **fast** rates
4. Compare how each configuration constrains the posterior distributions

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import time

import hdxrate
from hdxrate import MCMCConfig, process_single_protein
from hdxrate.types import TimepointCorrection
from hdxrate.testing import generate_mock_rows, recovery_summary
from hdxrate.spectra import group_rows_to_protein_group

print('hdxrate loaded from:', hdxrate.__file__)

## Shared experimental setup

All three sections use the same sequence, timepoints, and ground-truth rates so that
results are directly comparable. Only the set of pH conditions changes.

In [ ]:
SEQUENCE       = "ACDEFGHIKLMNQRSTVWY"
TIMEPOINTS     = np.array([10., 60., 300., 1800., 7200., 86400.])
N_EXCH         = sum(1 for i, aa in enumerate(SEQUENCE) if i >= 2 and aa != 'P')
TRUE_LOG_RATES = np.linspace(-10, -3, N_EXCH)

D2O_FRACTION   = 0.95
D2O_PURITY     = 0.95
TEMPERATURE    = 295.0
CHARGE         = 3
BACKEXCHANGE   = 0.20
NUM_BINS       = 30

# MCMC config — suitable for demo; use num_warmup=500, num_samples=1000 for publication
CFG = MCMCConfig(num_chains=2, num_warmup=150, num_samples=300, rng_seed=0)

print(f"Sequence: {SEQUENCE}")
print(f"Exchangeable residues: {N_EXCH}")
print(f"True log(k) range: [{TRUE_LOG_RATES[0]:.1f}, {TRUE_LOG_RATES[-1]:.1f}]")
print(f"Timepoints: {TIMEPOINTS}")

### Note on the saturation guard

For multi-pH experiments, `process_single_protein` checks whether the lowest-pH
series reached deuterium saturation — if not, backexchange estimation is unreliable
and a cross-protein correction is required.

In a real experiment you would call `process_hdx_data(rows)` on a multi-protein
dataset; it computes the cross-protein correction automatically and passes it to
each protein. For this notebook we pass a **no-op correction** that satisfies the
guard without altering backexchange — appropriate because the synthetic data has
a constant, known backexchange.

In [ ]:
# No-op correction: no actual timepoints match the sentinel (0.0),
# so apply_timepoint_correction returns the plain backexchange_value for every timepoint.
NO_CORRECTION = TimepointCorrection(
    timepoints=np.array([0.0]),
    correction_values=np.array([0.0]),
)

---
## 1. Single pH — pH 6.0

The simplest case: one set of timepoints, one MCMC pass, rates inferred directly
from the isotope envelope shift over time. No merge factor — `HDXResult.merge_factor`
is `None`.

```
generate_mock_rows(pHs=[6.0])  →  InputRows
process_single_protein(rows)   →  HDXResult  (no merge factor)
```

In [ ]:
rows_1ph = generate_mock_rows(
    sequence=SEQUENCE,
    true_log_rates=TRUE_LOG_RATES,
    timepoints=TIMEPOINTS,
    pHs=[6.0],
    charge=CHARGE,
    d2o_fraction=D2O_FRACTION,
    d2o_purity=D2O_PURITY,
    temperature=TEMPERATURE,
    true_backexchange=BACKEXCHANGE,
    num_bins=NUM_BINS,
    rng_seed=42,
)
print(f"Generated {len(rows_1ph)} rows  |  pHs: {sorted({r.pH for r in rows_1ph})}")

In [ ]:
t0 = time.time()
result_1ph = process_single_protein(
    rows=rows_1ph,
    mcmc_config=CFG,
    compute_free_energies=True,
)
print(f"1-pH  |  {time.time()-t0:.1f} s  |  RMSE = {result_1ph.rmse_total:.4f}  "
      f"|  merge_factor = {result_1ph.merge_factor}")

In [ ]:
pg_1ph  = group_rows_to_protein_group(rows_1ph)
exp_1ph = pg_1ph.distributions_by_pH[6.0]
tps_1ph = pg_1ph.timepoints_by_pH[6.0]

fig = hdxrate.plots.plot_isotope_fits_with_exp(
    exp_distributions=exp_1ph,
    result=result_1ph,
    timepoints=tps_1ph,
    max_cols=6,
)
fig.suptitle('1-pH: isotope fits at pH 6.0', y=1.01, fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
fig = hdxrate.plots.plot_posterior_rates(result_1ph)
fig.suptitle('1-pH: posterior rates (pH 6.0 only)', y=1.01, fontsize=10)
plt.tight_layout()
plt.show()

s1 = recovery_summary(result_1ph, TRUE_LOG_RATES)
print(f"Rate recovery:  MAE = {s1['mae']:.3f}  |  90%-coverage = {s1['coverage_90pct']:.0%}")

**Observation:** Wide violins on the slowest rates (left) — those residues barely
exchange within 86400 s at pH 6. Wide violins on the fastest rates (right) —
exchange is complete by the first timepoint (10 s). Both extremes are unconstrained.

Adding a second pH condition extends the effective time window in one or both
directions.

---
## 2. Two pH conditions — pH 6.0 + pH 9.0

pH 9 exchanges ~1000× faster than pH 6. The model infers a single **merge factor**
that relates the two timescales. A pH9 measurement at time *t* is equivalent to a
pH6 measurement at time *t × 10^mf*:

$$t^{\mathrm{eff},\mathrm{pH9}} = t^\mathrm{actual} \times 10^{+\mathrm{mf}}$$

**Prior:** `mf ~ TruncatedNormal(loc=3.0, scale=0.05, low=0)` — the prior mean 3.0
equals |9 − 6|, reflecting first-order base catalysis.

```
generate_mock_rows(pHs=[6.0, 9.0], true_merge_factors=3.0)  →  InputRows
process_single_protein(rows, backexchange_correction=...)    →  HDXResult
    result.merge_factor.mean.shape == (1,)                    # one merge factor
```

In [ ]:
TRUE_MF_2PH = 3.0   # true merge factor: pH9 exchanges 10^3 faster than pH6

rows_2ph = generate_mock_rows(
    sequence=SEQUENCE,
    true_log_rates=TRUE_LOG_RATES,
    timepoints=TIMEPOINTS,
    pHs=[6.0, 9.0],
    true_merge_factors=TRUE_MF_2PH,
    charge=CHARGE,
    d2o_fraction=D2O_FRACTION,
    d2o_purity=D2O_PURITY,
    temperature=TEMPERATURE,
    true_backexchange=BACKEXCHANGE,
    num_bins=NUM_BINS,
    rng_seed=42,
)
print(f"Generated {len(rows_2ph)} rows  |  pHs: {sorted({r.pH for r in rows_2ph})}")

In [ ]:
t0 = time.time()
result_2ph = process_single_protein(
    rows=rows_2ph,
    mcmc_config=CFG,
    backexchange_correction=NO_CORRECTION,
    compute_free_energies=True,
)
print(f"2-pH  |  {time.time()-t0:.1f} s  |  RMSE = {result_2ph.rmse_total:.4f}")

In [ ]:
mf2 = result_2ph.merge_factor
print(f"Merge factor — pH9 vs pH6 (reference)")
print(f"  True value   : {TRUE_MF_2PH:.2f}")
print(f"  Posterior mean: {mf2.mean[0]:.3f}")
print(f"  90% CI        : [{mf2.ci_5[0]:.3f}, {mf2.ci_95[0]:.3f}]")
print(f"  Std           : {mf2.std[0]:.4f}")

In [ ]:
# Visualise how the merge factor aligns pH9 uptake onto the pH6 timescale
pg_2ph   = group_rows_to_protein_group(rows_2ph)

def centroid(dist):
    idx = np.arange(len(dist))
    s = dist.sum()
    return float(np.dot(idx, dist) / s) if s > 0 else 0.0

uptake_6  = [centroid(pg_2ph.distributions_by_pH[6.0][i]) for i in range(len(TIMEPOINTS))]
uptake_9  = [centroid(pg_2ph.distributions_by_pH[9.0][i]) for i in range(len(TIMEPOINTS))]
eff_tps_9 = TIMEPOINTS * (10 ** mf2.mean[0])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.semilogx(TIMEPOINTS, uptake_6, 'o-', color='steelblue',  label='pH 6.0')
ax1.semilogx(TIMEPOINTS, uptake_9, 's--', color='darkorange', label='pH 9.0')
ax1.set_xlabel('Actual timepoint (s)', fontsize=9)
ax1.set_ylabel('Isotope centroid (a.u.)', fontsize=9)
ax1.set_title('Actual timescale', fontsize=9)
ax1.legend(fontsize=8)
ax1.spines[['top', 'right']].set_visible(False)

ax2.semilogx(TIMEPOINTS, uptake_6,  'o-',  color='steelblue',
             label='pH 6.0 (unchanged)')
ax2.semilogx(eff_tps_9,  uptake_9,  's--', color='darkorange',
             label=f'pH 9.0 (×10^{mf2.mean[0]:.2f} = effective pH6 scale)')
ax2.set_xlabel('Effective timepoint (s, pH 6 scale)', fontsize=9)
ax2.set_title('Aligned via merge factor', fontsize=9)
ax2.legend(fontsize=8)
ax2.spines[['top', 'right']].set_visible(False)

plt.suptitle('Merge factor maps pH 9 measurements onto pH 6 timescale', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
fig = hdxrate.plots.plot_posterior_rates(result_2ph)
fig.suptitle('2-pH: posterior rates (pH 6 + pH 9)', y=1.01, fontsize=10)
plt.tight_layout()
plt.show()

s2 = recovery_summary(result_2ph, TRUE_LOG_RATES)
print(f"Rate recovery:  MAE = {s2['mae']:.3f}  |  90%-coverage = {s2['coverage_90pct']:.0%}")

---
## 3. Three pH conditions — pH 5.0 + pH 6.0 + pH 9.0

Adding pH 5.0 (10× slower than pH 6) introduces a **second merge factor**:

$$t^{\mathrm{eff},\mathrm{pH5}} = t^\mathrm{actual} \times 10^{-\mathrm{mf}_5}$$

(note the **negative** sign — pH5 is below reference, so measurements at time *t*
correspond to a shorter effective exposure on the pH6 scale).

**Priors** (auto-derived from pH difference):
- `mf_pH5 ~ TruncatedNormal(loc=1.0, scale=0.05, low=0)` — |5 − 6| = 1
- `mf_pH9 ~ TruncatedNormal(loc=3.0, scale=0.05, low=0)` — |9 − 6| = 3

`merge_factor.mean` is ordered by **sorted non-reference pHs**: `[pH5, pH9]`.

```
generate_mock_rows(pHs=[5.0, 6.0, 9.0], true_merge_factors=[1.0, 3.0])
process_single_protein(rows, reference_pH=6.0, ...)  →  HDXResult
    result.merge_factor.mean.shape == (2,)             # two merge factors
```

In [ ]:
TRUE_MF_3PH = [1.0, 3.0]   # [pH5 vs pH6, pH9 vs pH6]

rows_3ph = generate_mock_rows(
    sequence=SEQUENCE,
    true_log_rates=TRUE_LOG_RATES,
    timepoints=TIMEPOINTS,
    pHs=[5.0, 6.0, 9.0],
    true_merge_factors=TRUE_MF_3PH,
    charge=CHARGE,
    d2o_fraction=D2O_FRACTION,
    d2o_purity=D2O_PURITY,
    temperature=TEMPERATURE,
    true_backexchange=BACKEXCHANGE,
    num_bins=NUM_BINS,
    rng_seed=42,
)
print(f"Generated {len(rows_3ph)} rows  |  pHs: {sorted({r.pH for r in rows_3ph})}")

In [ ]:
t0 = time.time()
result_3ph = process_single_protein(
    rows=rows_3ph,
    mcmc_config=CFG,
    backexchange_correction=NO_CORRECTION,
    reference_pH=6.0,
    compute_free_energies=True,
)
print(f"3-pH  |  {time.time()-t0:.1f} s  |  RMSE = {result_3ph.rmse_total:.4f}")

In [ ]:
mf3 = result_3ph.merge_factor
print("Merge factors — reference = pH 6.0")
print(f"  {'pH':>5}  {'True':>6}  {'Inferred':>10}  {'90% CI':>16}  {'Std':>6}")
for i, (pH, true_val) in enumerate([(5.0, TRUE_MF_3PH[0]), (9.0, TRUE_MF_3PH[1])]):
    print(f"  {pH:>5.1f}  {true_val:>6.2f}  {mf3.mean[i]:>10.3f}  "
          f"[{mf3.ci_5[i]:.3f}, {mf3.ci_95[i]:.3f}]  {mf3.std[i]:>6.4f}")

In [ ]:
fig = hdxrate.plots.plot_posterior_rates(result_3ph)
fig.suptitle('3-pH: posterior rates (pH 5 + pH 6 + pH 9)', y=1.01, fontsize=10)
plt.tight_layout()
plt.show()

s3 = recovery_summary(result_3ph, TRUE_LOG_RATES)
print(f"Rate recovery:  MAE = {s3['mae']:.3f}  |  90%-coverage = {s3['coverage_90pct']:.0%}")
print("(Use num_warmup=500, num_samples=1000 for tighter posteriors in publication-quality runs)")

---
## 4. Comparison: how many pH conditions do you need?

The same ground-truth rates, inferred under each configuration. Points on the
diagonal = perfect recovery. Tighter error bars = better-constrained rates.

In [ ]:
configs = [
    ('1-pH  (pH 6)',       result_1ph, 'steelblue'),
    ('2-pH  (pH 6+9)',     result_2ph, 'darkorange'),
    ('3-pH  (pH 5+6+9)',   result_3ph, 'forestgreen'),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)

for ax, (label, result, color) in zip(axes, configs):
    pr = result.posterior_rates
    n  = min(len(TRUE_LOG_RATES), len(pr.mean))
    true = TRUE_LOG_RATES[:n]
    mean = pr.mean[:n]
    lo   = pr.ci_5[:n]
    hi   = pr.ci_95[:n]

    ax.errorbar(true, mean,
                yerr=[mean - lo, hi - mean],
                fmt='o', ms=4, lw=0.7, color=color,
                ecolor=color, alpha=0.8, label='mean ± 90% CI')
    lim = (-11, -2)
    ax.plot(lim, lim, 'k--', lw=0.8, alpha=0.4)
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_xlabel('True log(k)', fontsize=9)
    ax.set_title(label, fontsize=9)
    ax.set_aspect('equal')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(alpha=0.2)

axes[0].set_ylabel('Inferred log(k)', fontsize=9)
plt.suptitle('Rate recovery across pH configurations (same ground truth)', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
print(f"{'Configuration':<22}  {'MAE':>6}  {'90% cov':>8}  {'Median CI width':>16}")
print('-' * 60)
for label, result, _ in configs:
    s   = recovery_summary(result, TRUE_LOG_RATES)
    pr  = result.posterior_rates
    n   = s['n_rates']
    ciw = float(np.median(pr.ci_95[:n] - pr.ci_5[:n]))
    print(f"{label:<22}  {s['mae']:>6.3f}  {s['coverage_90pct']:>7.0%}  {ciw:>16.2f}")

## Summary

| Configuration | Model | Free parameters |
|--------------|-------|----------------|
| 1-pH (pH 6) | `hdx_rate_model` | N rates + σ |
| 2-pH (pH 6 + 9) | `hdx_rate_model_with_merge` | N rates + 1 merge factor + σ |
| 3-pH (pH 5 + 6 + 9) | `hdx_rate_model_with_merge` | N rates + 2 merge factors + σ |

Key takeaways:
- **1-pH**: simplest, but slow and fast extremes are poorly constrained
- **2-pH** (+pH 9): adds data that constrains slow rates via extended effective time
- **3-pH** (+pH 5): adds data that constrains fast rates; both merge factors are
  inferred jointly and well-recovered
- `reference_pH` (default: pH closest to 6.0) anchors the rate scale; all other
  conditions are related through merge factors inferred from the data

**Production workflow** for multi-pH data:
```python
from hdxrate import process_hdx_data

# rows contains InputRow objects for multiple proteins, each measured at multiple pHs
results = process_hdx_data(
    rows=rows,
    mcmc_config=MCMCConfig(num_warmup=500, num_samples=1000),
    timepoint_correction=True,  # cross-protein backexchange correction (default)
)
# results is dict[protein_id → HDXResult]
```